In [1]:
library(Seurat)
library(tidyverse)

The legacy packages maptools, rgdal, and rgeos, underpinning the sp package,
which was just loaded, will retire in October 2023.
Please refer to R-spatial evolution reports for details, especially
https://r-spatial.org/r/2023/05/15/evolution4.html.
It may be desirable to make the sf package available;
package maintainers should consider adding sf to Suggests:.
The sp package is now running under evolution status 2
     (status 2 uses the sf package in place of rgdal)

Attaching SeuratObject

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.3     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all confli

In [5]:
humanMarkers.colon_polyps.stem.ensg <- c("ENSG00000165078", "ENSG00000139292", "ENSG00000140807", "ENSG00000112562", "ENSG00000183579")
humanMarkers.colon_polyps.goblet.ensg <- c("ENSG00000064787", "ENSG00000275395", "ENSG00000214814", "ENSG00000156049", "ENSG00000198788")

mouseMarkers.stomach_goblet <- c("Tff3", "Agr2", "Fcgbp", "Spink4", "Muc2")
humanMarkers.stomach_goblet <- c("TFF3", "SPINK4", "REG4", "FCGBP", "MUC2")

mouseMarkers.pancreas_acinar <- c("Cela2a", "Rnase1", "Zg16", "Cpa1", "Try4")
humanMarkers.pancreas_acinar <- c("PNLIPRP1", "PRSS1", "CELA3A", "RBPJL", "GP2")

# Tosti, L. et al., Gastroenterology 2021

In [2]:
tosti.cp <- readRDS("./data/rdsFiles/tosti_pancreas.epi.chronic_pancreatitis.rds")

tosti.cp
dim(tosti.cp@meta.data)
head(tosti.cp@meta.data)

An object of class Seurat 
26782 features across 1161 samples within 2 assays 
Active assay: RNA (24782 features, 0 variable features)
 1 other assay present: integrated
 1 dimensional reduction calculated: umap

[1] 1161   11

,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,sample_ID,Cluster,patient_ID,sex,age,hla.acinar,ann.epi
,<chr>,<dbl>,<int>,<dbl>,<chr>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>
AAACGAACACTCATAG_1,pancreatitis,8928,4183,0.6720430,TUM_CP1,Ductal,TUM_CP1,male,71,Ductal,Epithelial
AAACGAAGTATTCTCT_1,pancreatitis,11008,4826,2.4345930,TUM_CP1,MUC5B+ Ductal,TUM_CP1,male,71,MUC5B+ Ductal,Epithelial
AAACGAAGTTCTTAGG_1,pancreatitis,13008,4386,0.5381304,TUM_CP1,Beta,TUM_CP1,male,71,Beta,Epithelial
AAACGAAGTTGCGTAT_1,pancreatitis,4423,2372,0.6104454,TUM_CP1,Tuft,TUM_CP1,male,71,Tuft,Epithelial
AAACGAATCAAGAATG_1,pancreatitis,9526,4010,1.0392610,TUM_CP1,Beta,TUM_CP1,male,71,Beta,Epithelial
AAAGAACCACTCACTC_1,pancreatitis,8710,4231,2.0206659,TUM_CP1,MUC5B+ Ductal,TUM_CP1,male,71,MUC5B+ Ductal,Epithelial


In [22]:
print(all(humanMarkers.stomach_goblet %in% rownames(tosti.cp)))
print(humanMarkers.stomach_goblet[!humanMarkers.stomach_goblet %in% rownames(tosti.cp)])

print(all(humanMarkers.pancreas_acinar %in% rownames(tosti.cp)))
print(humanMarkers.pancreas_acinar[!humanMarkers.pancreas_acinar %in% rownames(tosti.cp)])

[1] FALSE
[1] "MUC2"
[1] TRUE
character(0)


In [23]:
tosti.cp <- AddModuleScore(object = tosti.cp, 
                             features = list(humanMarkers.pancreas_acinar, humanMarkers.stomach_goblet), 
                             name = c("humanMarkers.pancreas_acinar", "humanMarkers.stomach_goblet"))

tosti.cp@meta.data <- tosti.cp@meta.data |>
    rename("humanMarkers.pancreas_acinar" = "humanMarkers.pancreas_acinar1", 
           "humanMarkers.stomach_goblet" = "humanMarkers.stomach_goblet2")
head(tosti.cp@meta.data)

Warning message:
“The following features are not present in the object: MUC2, not searching for symbol synonyms”


,orig.ident,nCount_RNA,nFeature_RNA,percent.mt,sample_ID,Cluster,patient_ID,sex,age,hla.acinar,ann.epi,humanMarkers.pancreas_acinar,humanMarkers.stomach_goblet
,<chr>,<dbl>,<int>,<dbl>,<chr>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
AAACGAACACTCATAG_1,pancreatitis,8928,4183,0.6720430,TUM_CP1,Ductal,TUM_CP1,male,71,Ductal,Epithelial,-0.39388247,-0.009794140
AAACGAAGTATTCTCT_1,pancreatitis,11008,4826,2.4345930,TUM_CP1,MUC5B+ Ductal,TUM_CP1,male,71,MUC5B+ Ductal,Epithelial,-0.32572252,-0.024258857
AAACGAAGTTCTTAGG_1,pancreatitis,13008,4386,0.5381304,TUM_CP1,Beta,TUM_CP1,male,71,Beta,Epithelial,-0.08643434,-0.077529364
AAACGAAGTTGCGTAT_1,pancreatitis,4423,2372,0.6104454,TUM_CP1,Tuft,TUM_CP1,male,71,Tuft,Epithelial,-0.08257147,-0.307020451
AAACGAATCAAGAATG_1,pancreatitis,9526,4010,1.0392610,TUM_CP1,Beta,TUM_CP1,male,71,Beta,Epithelial,-0.46408301,-0.008451394
AAAGAACCACTCACTC_1,pancreatitis,8710,4231,2.0206659,TUM_CP1,MUC5B+ Ductal,TUM_CP1,male,71,MUC5B+ Ductal,Epithelial,-0.23525638,0.191060892


In [33]:
saveRDS(tosti.cp, "./data/rdsFiles/tosti_pancreas.epi.chronic_pancreatitis.rds")

# Ma, Z. et al., Gastroenterology 2022

In [3]:
ma.pinjury <- readRDS("./data/rdsFiles/ma_pancreas_injury.yfpPositive.rds")

ma.pinjury
dim(ma.pinjury@meta.data)
head(ma.pinjury@meta.data)

An object of class Seurat 
32288 features across 13362 samples within 1 assay 
Active assay: RNA (32288 features, 2000 variable features)
 2 dimensional reductions calculated: pca, umap

[1] 13362     6

,orig.ident,nCount_RNA,nFeature_RNA,clusters,celltypeLabel,cellBarcode
,<fct>,<dbl>,<int>,<dbl>,<chr>,<chr>
AAACCCACACTCACTC.2wks.A8324,SeuratProject,16196,4794,11,EEC,AAACCCACACTCACTC-2wks-A8324
AAACCCACAGGCATTT.2wks.A8324,SeuratProject,47452,6743,2,MucinDuctal,AAACCCACAGGCATTT-2wks-A8324
AAACCCAGTGAGTTTC.2wks.A8324,SeuratProject,22929,5617,10,Tuft,AAACCCAGTGAGTTTC-2wks-A8324
AAACCCATCCTTATAC.2wks.A8324,SeuratProject,9305,1419,0,Acinar,AAACCCATCCTTATAC-2wks-A8324
AAACCCATCGCTCCTA.2wks.A8324,SeuratProject,29731,5101,2,MucinDuctal,AAACCCATCGCTCCTA-2wks-A8324
AAACGAACAGGAAGTC.2wks.A8324,SeuratProject,66120,7282,2,MucinDuctal,AAACGAACAGGAAGTC-2wks-A8324


In [21]:
print(all(mouseMarkers.stomach_goblet %in% rownames(ma.pinjury)))
print(mouseMarkers.stomach_goblet[!mouseMarkers.stomach_goblet %in% rownames(ma.pinjury)])

print(all(mouseMarkers.pancreas_acinar %in% rownames(ma.pinjury)))
print(mouseMarkers.pancreas_acinar[!mouseMarkers.pancreas_acinar %in% rownames(ma.pinjury)])

[1] TRUE
character(0)
[1] TRUE
character(0)


In [7]:
ma.pinjury <- AddModuleScore(object = ma.pinjury, 
                             features = list(mouseMarkers.pancreas_acinar, mouseMarkers.stomach_goblet), 
                             name = c("mouseMarkers.pancreas_acinar", "mouseMarkers.stomach_goblet"))

ma.pinjury@meta.data <- ma.pinjury@meta.data |>
    rename("mouseMarkers.pancreas_acinar" = "mouseMarkers.pancreas_acinar1", 
           "mouseMarkers.stomach_goblet" = "mouseMarkers.stomach_goblet2")
head(ma.pinjury@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,clusters,celltypeLabel,cellBarcode,mouseMarkers.pancreas_acinar,mouseMarkers.stomach_goblet
,<fct>,<dbl>,<int>,<dbl>,<chr>,<chr>,<dbl>,<dbl>
AAACCCACACTCACTC.2wks.A8324,SeuratProject,16196,4794,11,EEC,AAACCCACACTCACTC-2wks-A8324,-0.5193740,-0.3969829
AAACCCACAGGCATTT.2wks.A8324,SeuratProject,47452,6743,2,MucinDuctal,AAACCCACAGGCATTT-2wks-A8324,-0.5636725,0.5269775
AAACCCAGTGAGTTTC.2wks.A8324,SeuratProject,22929,5617,10,Tuft,AAACCCAGTGAGTTTC-2wks-A8324,0.2204598,-0.3865462
AAACCCATCCTTATAC.2wks.A8324,SeuratProject,9305,1419,0,Acinar,AAACCCATCCTTATAC-2wks-A8324,3.7854557,-0.2443634
AAACCCATCGCTCCTA.2wks.A8324,SeuratProject,29731,5101,2,MucinDuctal,AAACCCATCGCTCCTA-2wks-A8324,-0.3075051,1.0477709
AAACGAACAGGAAGTC.2wks.A8324,SeuratProject,66120,7282,2,MucinDuctal,AAACGAACAGGAAGTC-2wks-A8324,-0.7085076,0.9901325


In [32]:
saveRDS(ma.pinjury, "./data/rdsFiles/ma_pancreas_injury.yfpPositive.rds")

# Burdziak, C. et al., Science 2023

In [4]:
burdziak.pdac <- readRDS("./data/rdsFiles/burdziak_pdac_models.progressionCohort.rds")

burdziak.pdac
dim(burdziak.pdac@meta.data)
head(burdziak.pdac@meta.data)

An object of class Seurat 
16826 features across 28131 samples within 1 assay 
Active assay: RNA (16826 features, 0 variable features)
 2 dimensional reductions calculated: EMBED, PCA

[1] 28131     3

,batch,condition,cluster
,<fct>,<fct>,<dbl>
120703424252788_DACD394_Kate_plus,DACD394_Kate_plus,K1,0
120703436605797_DACD394_Kate_plus,DACD394_Kate_plus,K1,1
120703436909853_DACD394_Kate_plus,DACD394_Kate_plus,K1,1
120703454952236_DACD394_Kate_plus,DACD394_Kate_plus,K1,1
120726896892262_DACD394_Kate_plus,DACD394_Kate_plus,K1,5
120726897153838_DACD394_Kate_plus,DACD394_Kate_plus,K1,0


In [19]:
print(all(mouseMarkers.stomach_goblet %in% rownames(burdziak.pdac)))
print(mouseMarkers.stomach_goblet[!mouseMarkers.stomach_goblet %in% rownames(burdziak.pdac)])

print(all(mouseMarkers.pancreas_acinar %in% rownames(burdziak.pdac)))
print(mouseMarkers.pancreas_acinar[!mouseMarkers.pancreas_acinar %in% rownames(burdziak.pdac)])

[1] FALSE
[1] "Tff3"   "Agr2"   "Fcgbp"  "Spink4" "Muc2"  
[1] FALSE
[1] "Cela2a" "Rnase1" "Zg16"   "Cpa1"   "Try4"  


In [15]:
# We need to change the gene universe in this object to have gene symbols as uppercases
print(all(str_to_upper(mouseMarkers.stomach_goblet) %in% rownames(burdziak.pdac)))
print(str_to_upper(mouseMarkers.stomach_goblet)[!str_to_upper(mouseMarkers.stomach_goblet) %in% rownames(burdziak.pdac)])

print(all(str_to_upper(mouseMarkers.pancreas_acinar) %in% rownames(burdziak.pdac)))
print(str_to_upper(mouseMarkers.pancreas_acinar)[!str_to_upper(mouseMarkers.pancreas_acinar) %in% rownames(burdziak.pdac)])

[1] TRUE
character(0)
[1] TRUE
character(0)


In [16]:
burdziak.pdac <- AddModuleScore(object = burdziak.pdac, 
                                features = list(str_to_upper(mouseMarkers.pancreas_acinar), str_to_upper(mouseMarkers.stomach_goblet)), 
                                name = c("mouseMarkers.pancreas_acinar", "mouseMarkers.stomach_goblet"))

burdziak.pdac@meta.data <- burdziak.pdac@meta.data |>
    rename("mouseMarkers.pancreas_acinar" = "mouseMarkers.pancreas_acinar1", 
           "mouseMarkers.stomach_goblet" = "mouseMarkers.stomach_goblet2")
burdziak.pdac@meta.data

,batch,condition,cluster,mouseMarkers.pancreas_acinar,mouseMarkers.stomach_goblet
,<fct>,<fct>,<dbl>,<dbl>,<dbl>
120703424252788_DACD394_Kate_plus,DACD394_Kate_plus,K1,0,-2.7694225,1.440895002
120703436605797_DACD394_Kate_plus,DACD394_Kate_plus,K1,1,-1.0813835,-0.629677702
120703436909853_DACD394_Kate_plus,DACD394_Kate_plus,K1,1,-1.9626524,-0.265274470
120703454952236_DACD394_Kate_plus,DACD394_Kate_plus,K1,1,-1.6521811,-0.363092520
120726896892262_DACD394_Kate_plus,DACD394_Kate_plus,K1,5,-1.6916516,-1.029222239
120726897153838_DACD394_Kate_plus,DACD394_Kate_plus,K1,0,-1.9432092,0.772819929
120726924409269_DACD394_Kate_plus,DACD394_Kate_plus,K1,3,3.4148721,-0.007626893
120726924679596_DACD394_Kate_plus,DACD394_Kate_plus,K1,0,-2.0782286,0.454381602
120726943616435_DACD394_Kate_plus,DACD394_Kate_plus,K1,1,-2.7782229,-0.577349450


In [31]:
saveRDS(burdziak.pdac, "./data/rdsFiles/burdziak_pdac_models.progressionCohort.rds")

# Chen, B. et al., Cell 2021

In [18]:
chen.dis_val <- readRDS("./data/rdsFiles/chen.precancer_crc_states.epi.dis_val.asc_ssc.rds")
chen.dis_val
dim(chen.dis_val@meta.data)
head(chen.dis_val@meta.data)

An object of class Seurat 
34614 features across 19312 samples within 1 assay 
Active assay: RNA (34614 features, 0 variable features)

[1] 19312    26

,HTAN Parent Data File ID,HTAN Specimen ID,Cell_Type,Polyp_Type,Sample_Classification,cell_type_ontology_term_id,disease_ontology_term_id,donor_id,assay_ontology_term_id,tissue_ontology_term_id,⋯,cell_type,assay,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,nCount_RNA,nFeature_RNA
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>
DIS_GACTTCTTCGATATGCAT-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,13449,3174
DIS_GAACCACGCTACCTTGCC-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,3473,1227
DIS_TGCCTCACGTGGAGCT-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,10235,2465
DIS_TGAACTAGCCAGGAATAGA-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,8249,2701
DIS_AGGGAACGAAGGCAACG-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,3894,1353
DIS_AACCTGACAGCAGAAC-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,neoplastic cell,single-cell RNA sequencing,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,5330,1606


In [28]:
print(all(humanMarkers.colon_polyps.stem.ensg %in% rownames(chen.dis_val)))
print(humanMarkers.colon_polyps.stem.ensg[!humanMarkers.colon_polyps.stem.ensg %in% rownames(chen.dis_val)])

print(all(humanMarkers.colon_polyps.goblet.ensg %in% rownames(chen.dis_val)))
print(humanMarkers.colon_polyps.goblet.ensg[!humanMarkers.colon_polyps.goblet.ensg %in% rownames(chen.dis_val)])

[1] TRUE
character(0)
[1] TRUE
character(0)


In [29]:
chen.dis_val <- AddModuleScore(object = chen.dis_val, 
                             features = list(humanMarkers.colon_polyps.stem.ensg, humanMarkers.colon_polyps.goblet.ensg), 
                             name = c("humanMarkers.colon_polyps.stem.ensg","humanMarkers.colon_polyps.goblet.ensg"))

chen.dis_val@meta.data <- chen.dis_val@meta.data |>
    rename("humanMarkers.colon_polyps.stem.ensg" = "humanMarkers.colon_polyps.stem.ensg1", 
           "humanMarkers.colon_polyps.goblet.ensg" = "humanMarkers.colon_polyps.goblet.ensg2")
head(chen.dis_val@meta.data)

,HTAN Parent Data File ID,HTAN Specimen ID,Cell_Type,Polyp_Type,Sample_Classification,cell_type_ontology_term_id,disease_ontology_term_id,donor_id,assay_ontology_term_id,tissue_ontology_term_id,⋯,disease,organism,sex,tissue,self_reported_ethnicity,development_stage,nCount_RNA,nFeature_RNA,humanMarkers.colon_polyps.stem.ensg,humanMarkers.colon_polyps.goblet.ensg
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<dbl>,<dbl>
DIS_GACTTCTTCGATATGCAT-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,13449,3174,-0.47939691,-0.5431514
DIS_GAACCACGCTACCTTGCC-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,3473,1227,-0.02306254,-0.1664093
DIS_TGCCTCACGTGGAGCT-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,10235,2465,0.11833286,-0.4740813
DIS_TGAACTAGCCAGGAATAGA-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,8249,2701,0.05550029,-0.6648685
DIS_AGGGAACGAAGGCAACG-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,3894,1353,0.08652106,-0.5584206
DIS_AACCTGACAGCAGAAC-0,HTA11_3410_200000101113111,HTA11_3410_2000001011,ASC,TA,AD,CL:0001063,MONDO:0024660,HTA11_3410,EFO:0008913,UBERON:0001157,⋯,tubular adenoma,Homo sapiens,male,transverse colon,European,70-year-old human stage,5330,1606,0.88706272,-0.3430571


In [30]:
saveRDS(chen.dis_val, "./data/rdsFiles/chen.precancer_crc_states.epi.dis_val.asc_ssc.rds")